# Truth/reconstruction model development

This notebook is the primary workspace for feature construction, model development, training, scoring, and evaluation.

Truth and reconstructed samples use the same cells and differ only through `DOMAIN` and the sample paths in `configs/pipeline.toml`. Keep scientific changes visible here: when changing cuts, features, weighting, architecture, or evaluation, duplicate the notebook or use a new `RUN_NAME`.

## Workflow

1. Choose a domain and experiment name.
2. Load preprocessed wide Parquet samples.
3. Apply the common training selection.
4. Build high-level and constituent inputs.
5. Split and scale the data.
6. select or modify a model.
7. Train with weighted binary cross-entropy.
8. Inspect performance and save the run outputs.

The ROOT-to-Parquet step is handled by `scripts/preprocess.py`. The expected columns are documented in `docs/DATA_SCHEMA.md`.

In [ ]:
from contextlib import nullcontext
import json
from pathlib import Path
import random
import shutil
import tomllib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    auc,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

plt.style.use("seaborn-v0_8-whitegrid")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## Experiment settings

These are the main controls for an exploratory run. `MODEL_NAME` may be `dnn`, `gnn`, `gdnn`, `transformer`, `tdnn`, `deepsets`, or `hybrid_deepsets`.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "configs/pipeline.toml"
DOMAIN = "reco"                 # "reco" or "truth"
MODEL_NAME = "gdnn"
RUN_NAME = "baseline_v1"

with CONFIG_PATH.open("rb") as stream:
    CONFIG = tomllib.load(stream)

PREPROCESSING = CONFIG["preprocessing"]
SELECTION = CONFIG["selection"]
TRAINING = CONFIG["training"]

SEED = int(TRAINING["seed"])
MAX_TRACKS = int(PREPROCESSING["max_tracks"])
BATCH_SIZE = int(TRAINING["batch_size"])

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Domain={DOMAIN}  Model={MODEL_NAME}  Run={RUN_NAME}")

## Load processed samples

Each configured sample receives explicit `sample`, `domain`, and `label` columns here. This keeps file metadata authoritative even if a Parquet file was moved or created by an older preprocessor.

In [ ]:
def resolve_project_path(value):
    path = Path(value)
    return path if path.is_absolute() else PROJECT_ROOT / path


sample_frames = []
selected_sample_config = {
    name: sample
    for name, sample in CONFIG["samples"].items()
    if sample["domain"] == DOMAIN
}

if {sample["label"] for sample in selected_sample_config.values()} != {0, 1}:
    raise ValueError(f"Domain {DOMAIN!r} must contain signal and background samples.")

for sample_name, sample in selected_sample_config.items():
    path = resolve_project_path(sample["output"])
    if not path.exists():
        raise FileNotFoundError(f"Run preprocessing first; missing {path}")
    frame = pd.read_parquet(path)
    frame["sample"] = sample_name
    frame["domain"] = sample["domain"]
    frame["label"] = int(sample["label"])
    sample_frames.append(frame)
    print(sample_name, len(frame), path)

## Physics helpers and common selection

The loose cuts in preprocessing reduce I/O. The tighter cuts below define the event population used for model comparisons and therefore should be kept identical between truth and reconstruction unless the study explicitly tests a selection change.

In [ ]:
def delta_phi(phi1, phi2):
    return (np.asarray(phi1) - np.asarray(phi2) + np.pi) % (2 * np.pi) - np.pi


def delta_r(eta1, eta2, phi1, phi2):
    return np.hypot(np.asarray(eta1) - np.asarray(eta2), delta_phi(phi1, phi2))


def derive_angular_features(frame):
    frame = frame.copy()
    pairs = (
        ("muon_fatjet", "muonEta", "fatJetEta", "muonPhi", "fatJetPhi"),
        ("muon_jet1", "muonEta", "jet1Eta", "muonPhi", "jet1Phi"),
        ("muon_jet2", "muonEta", "jet2Eta", "muonPhi", "jet2Phi"),
    )
    for prefix, eta1, eta2, phi1, phi2 in pairs:
        if {eta1, eta2, phi1, phi2}.issubset(frame.columns):
            frame[f"{prefix}_deta"] = frame.get(f"{prefix}_deta", frame[eta1] - frame[eta2])
            frame[f"{prefix}_dphi"] = frame.get(
                f"{prefix}_dphi", delta_phi(frame[phi1], frame[phi2])
            )
            frame[f"{prefix}_dr"] = frame.get(
                f"{prefix}_dr", delta_r(frame[eta1], frame[eta2], frame[phi1], frame[phi2])
            )
    if {"muonPhi", "METPhi"}.issubset(frame.columns):
        frame["muon_MET_dphi"] = frame.get(
            "muon_MET_dphi", delta_phi(frame["muonPhi"], frame["METPhi"])
        )
    if {"fatJetPhi", "METPhi"}.issubset(frame.columns):
        frame["fatjet_MET_dphi"] = frame.get(
            "fatjet_MET_dphi", delta_phi(frame["fatJetPhi"], frame["METPhi"])
        )
    return frame


def apply_selection(frame):
    frame = derive_angular_features(frame)
    required = {"fatJetPt", "fatJetM", "muon_fatjet_dr"}
    missing = required - set(frame.columns)
    if missing:
        raise KeyError("Missing selection columns: " + ", ".join(sorted(missing)))

    keep = (
        (frame["fatJetPt"] > float(SELECTION["fatjet_pt_min"]))
        & (frame["fatJetM"] > float(SELECTION["fatjet_mass_min"]))
        & (frame["muon_fatjet_dr"] < float(SELECTION["lepton_fatjet_dr_max"]))
    )
    selected = frame.loc[keep].copy()
    invalid = float(SELECTION["invalid_jet_value"])
    for column in ("jet1Pt", "jet2Pt"):
        if column in selected:
            selected = selected[selected[column] != invalid]
    return selected.reset_index(drop=True)


selected_frames = [apply_selection(frame) for frame in sample_frames]
for name, frame in zip(selected_sample_config, selected_frames):
    print(name, "selected events:", len(frame))

## High-level event features

The default list matches the later reconstructed analysis. Edit this cell directly for feature-ablation or feature-addition studies. Missing requested features are reported instead of silently replaced.

In [ ]:
DNN_FEATURES = [
    "numOfInDetTracks",
    "numOfFatJets",
    "numOfInDetTracks_inFJCone",
    "muonPt",
    "muonEta",
    "fatJetPt",
    "fatJetM",
    "fatJetD2",
    "fatJetEta",
    "jet1Pt",
    "jet1M",
    "jet2Pt",
    "jet2M",
    "MET",
    "muon_fatjet_dr",
    "muon_fatjet_deta",
    "muon_fatjet_dphi",
    "muon_jet1_dr",
    "muon_jet1_deta",
    "muon_jet1_dphi",
    "muon_jet2_dr",
    "muon_jet2_deta",
    "muon_jet2_dphi",
    "muon_MET_dphi",
    "fatjet_MET_dphi",
]

combined = pd.concat(selected_frames, ignore_index=True, sort=False)
missing_features = [name for name in DNN_FEATURES if name not in combined]
if missing_features:
    raise KeyError("Missing DNN features: " + ", ".join(missing_features))

X_DNN_RAW = combined[DNN_FEATURES].fillna(0).to_numpy(np.float32)
print("High-level tensor:", X_DNN_RAW.shape)

## Constituent features

The baseline uses four equivalent truth/reco inputs for the 30 retained tracks: relative transverse momentum, relative eta, wrapped relative phi, and delta-R. Optional truth identity columns remain in the Parquet files and can be added here for a dedicated truth-only study.

In [ ]:
PARTICLE_FEATURES = ["rel_pt", "deta", "dphi", "dr"]


def build_particle_tensor(frame, max_tracks=30):
    required = {"fatJetPt", "fatJetEta", "fatJetPhi"}
    missing = required - set(frame.columns)
    if missing:
        raise KeyError("Missing fat-jet parent columns: " + ", ".join(sorted(missing)))

    tensor = np.zeros((len(frame), max_tracks, 4), dtype=np.float32)
    fj_pt = frame["fatJetPt"].to_numpy(float)
    fj_eta = frame["fatJetEta"].to_numpy(float)
    fj_phi = frame["fatJetPhi"].to_numpy(float)

    for slot in range(max_tracks):
        number = slot + 1
        pt_name = f"fj_track{number}Pt"
        if pt_name not in frame:
            continue

        track_pt = frame[pt_name].fillna(0).to_numpy(float)
        real = track_pt > 0
        rel_pt = np.divide(track_pt, fj_pt, out=np.zeros_like(track_pt), where=fj_pt != 0)

        deta_name = f"fj_track{number}dEta"
        dphi_name = f"fj_track{number}dPhi"
        dr_name = f"fj_track{number}dR"
        eta_name = f"fj_track{number}Eta"
        phi_name = f"fj_track{number}Phi"

        if deta_name in frame:
            deta = frame[deta_name].fillna(0).to_numpy(float)
        elif eta_name in frame:
            deta = frame[eta_name].fillna(0).to_numpy(float) - fj_eta
        else:
            deta = np.zeros(len(frame))

        if dphi_name in frame:
            dphi = frame[dphi_name].fillna(0).to_numpy(float)
        elif phi_name in frame:
            dphi = delta_phi(frame[phi_name].fillna(0).to_numpy(float), fj_phi)
        else:
            dphi = np.zeros(len(frame))

        dr = frame[dr_name].fillna(0).to_numpy(float) if dr_name in frame else np.hypot(deta, dphi)
        tensor[:, slot] = np.column_stack(
            [np.where(real, values, 0) for values in (rel_pt, deta, dphi, dr)]
        )
    return tensor


X_PARTICLES_RAW = build_particle_tensor(combined, MAX_TRACKS)
has_particles = np.any(X_PARTICLES_RAW != 0, axis=(1, 2))
combined = combined.loc[has_particles].reset_index(drop=True)
X_DNN_RAW = X_DNN_RAW[has_particles]
X_PARTICLES_RAW = X_PARTICLES_RAW[has_particles]
print("Particle tensor:", X_PARTICLES_RAW.shape)
print("Events removed because no retained tracks:", int((~has_particles).sum()))

## Labels and event weights

Signal and total background receive equal aggregate weight. When several background productions are configured, each background sample first receives equal aggregate influence.

In [ ]:
Y = combined["label"].to_numpy(np.float32)


def make_event_weights(frame):
    labels = frame["label"].to_numpy(int)
    weights = np.ones(len(frame), dtype=np.float64)
    for label in np.unique(labels):
        class_mask = labels == label
        if label == 0:
            background_names = frame.loc[class_mask, "sample"].astype(str).unique()
            for sample_name in background_names:
                mask = class_mask & (frame["sample"].astype(str).to_numpy() == sample_name)
                weights[mask] = 1 / (len(background_names) * mask.sum())
        else:
            weights[class_mask] = 1 / class_mask.sum()

    for label in np.unique(labels):
        mask = labels == label
        weights[mask] *= (len(frame) / 2) / weights[mask].sum()
    return weights.astype(np.float32)


WEIGHTS = make_event_weights(combined)
pd.DataFrame({"label": Y, "weight": WEIGHTS}).groupby("label").agg(["count", "sum"])

## Stratified split and scaling

Scalers are fitted only on the training split. Particle padding remains exactly zero.

In [ ]:
all_indices = np.arange(len(Y))
train_indices, holdout_indices = train_test_split(
    all_indices,
    test_size=float(TRAINING["validation_fraction"]) + float(TRAINING["test_fraction"]),
    stratify=Y,
    random_state=SEED,
)
test_share = float(TRAINING["test_fraction"]) / (
    float(TRAINING["validation_fraction"]) + float(TRAINING["test_fraction"])
)
validation_indices, test_indices = train_test_split(
    holdout_indices,
    test_size=test_share,
    stratify=Y[holdout_indices],
    random_state=SEED,
)

dnn_scaler = StandardScaler().fit(X_DNN_RAW[train_indices])
X_DNN = dnn_scaler.transform(X_DNN_RAW).astype(np.float32)

particle_scaler = StandardScaler()
training_particles = X_PARTICLES_RAW[train_indices].reshape(-1, X_PARTICLES_RAW.shape[-1])
training_real = np.any(training_particles != 0, axis=1)
particle_scaler.fit(training_particles[training_real])

flat_particles = X_PARTICLES_RAW.reshape(-1, X_PARTICLES_RAW.shape[-1]).copy()
real_particles = np.any(flat_particles != 0, axis=1)
flat_particles[real_particles] = particle_scaler.transform(flat_particles[real_particles])
flat_particles[~real_particles] = 0
X_PARTICLES = flat_particles.reshape(X_PARTICLES_RAW.shape).astype(np.float32)

print("Split sizes:", len(train_indices), len(validation_indices), len(test_indices))

In [ ]:
def make_loader(indices, shuffle=False):
    dataset = TensorDataset(
        torch.from_numpy(X_PARTICLES[indices]),
        torch.from_numpy(X_DNN[indices]),
        torch.from_numpy(Y[indices]).view(-1, 1),
        torch.from_numpy(WEIGHTS[indices]).view(-1, 1),
        torch.from_numpy(indices),
    )
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=int(TRAINING["num_workers"]),
        pin_memory=torch.cuda.is_available(),
        drop_last=shuffle and len(dataset) > BATCH_SIZE and len(dataset) % BATCH_SIZE == 1,
    )


train_loader = make_loader(train_indices, shuffle=True)
validation_loader = make_loader(validation_indices)
test_loader = make_loader(test_indices)
full_loader = make_loader(all_indices)

## DNN and interaction-network models

All models use the same `forward(particles, event_features)` interface so the training cell remains editable but shared. Padded graph nodes and their edges are masked.

In [ ]:
class DNN(nn.Module):
    def __init__(self, event_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(event_features, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.15),
            nn.Linear(128, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.15),
            nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1),
        )

    def forward(self, particles, event_features):
        return self.network(event_features)


class InteractionEncoder(nn.Module):
    def __init__(self, max_particles, particle_features, latent_dim=24):
        super().__init__()
        receivers = np.zeros((max_particles, max_particles * (max_particles - 1)), np.float32)
        senders = np.zeros_like(receivers)
        edge = 0
        for receiver in range(max_particles):
            for sender in range(max_particles):
                if receiver == sender:
                    continue
                receivers[receiver, edge] = 1
                senders[sender, edge] = 1
                edge += 1
        self.register_buffer("receivers", torch.from_numpy(receivers))
        self.register_buffer("senders", torch.from_numpy(senders))
        self.relation = nn.Sequential(
            nn.Linear(2 * particle_features, 80), nn.ReLU(),
            nn.Linear(80, 50), nn.ReLU(),
            nn.Linear(50, 30), nn.ReLU(),
        )
        self.object = nn.Sequential(
            nn.Linear(particle_features + 30, 80), nn.ReLU(),
            nn.Linear(80, 50), nn.ReLU(),
            nn.Linear(50, latent_dim), nn.ReLU(),
        )

    def forward(self, particles):
        node_mask = torch.any(particles != 0, dim=-1)
        values = particles.transpose(1, 2)
        receivers = torch.matmul(values, self.receivers)
        senders = torch.matmul(values, self.senders)
        edges = torch.cat((receivers, senders), dim=1).transpose(1, 2)
        edge_mask = (
            (torch.matmul(node_mask.float(), self.receivers) > 0)
            & (torch.matmul(node_mask.float(), self.senders) > 0)
        )
        effects = self.relation(edges) * edge_mask.unsqueeze(-1)
        aggregated = torch.matmul(effects.transpose(1, 2), self.receivers.T).transpose(1, 2)
        nodes = self.object(torch.cat((particles, aggregated), dim=-1))
        return (nodes * node_mask.unsqueeze(-1)).sum(dim=1)


class GNN(nn.Module):
    def __init__(self, max_particles, particle_features):
        super().__init__()
        self.encoder = InteractionEncoder(max_particles, particle_features)
        self.classifier = nn.Sequential(
            nn.Linear(24, 60), nn.ReLU(), nn.BatchNorm1d(60),
            nn.Linear(60, 30), nn.ReLU(), nn.Linear(30, 10), nn.ReLU(), nn.Linear(10, 1),
        )

    def forward(self, particles, event_features):
        return self.classifier(self.encoder(particles))


class GDNN(nn.Module):
    def __init__(self, max_particles, particle_features, event_features):
        super().__init__()
        self.encoder = InteractionEncoder(max_particles, particle_features)
        self.classifier = DNN(event_features + 24).network

    def forward(self, particles, event_features):
        return self.classifier(torch.cat((self.encoder(particles), event_features), dim=1))

## Transformer and Deep Sets models

These cells are intentionally local to the notebook so researchers can change embedding sizes, attention depth, pooling, and classifier heads without modifying package internals.

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, particle_features, embedding_dim=128, heads=8, layers=5, latent_dim=24):
        super().__init__()
        self.embedding = nn.Linear(particle_features, embedding_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=heads,
            dim_feedforward=512,
            dropout=0.1,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers)
        self.projection = nn.Sequential(
            nn.Linear(embedding_dim, 64), nn.ReLU(), nn.Linear(64, latent_dim), nn.ReLU()
        )

    def forward(self, particles):
        padding = ~torch.any(particles != 0, dim=-1)
        encoded = self.encoder(self.embedding(particles), src_key_padding_mask=padding)
        real = (~padding).unsqueeze(-1)
        pooled = (encoded * real).sum(dim=1) / real.sum(dim=1).clamp_min(1)
        return self.projection(pooled)


class Transformer(nn.Module):
    def __init__(self, particle_features):
        super().__init__()
        self.encoder = TransformerEncoder(particle_features)
        self.classifier = nn.Sequential(nn.Linear(24, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, particles, event_features):
        return self.classifier(self.encoder(particles))


class TDNN(nn.Module):
    def __init__(self, particle_features, event_features):
        super().__init__()
        self.encoder = TransformerEncoder(particle_features)
        self.classifier = DNN(event_features + 24).network

    def forward(self, particles, event_features):
        return self.classifier(torch.cat((self.encoder(particles), event_features), dim=1))


class DeepSetsEncoder(nn.Module):
    def __init__(self, particle_features, latent_dim=24):
        super().__init__()
        self.phi = nn.Sequential(
            nn.Linear(particle_features, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU()
        )
        self.rho = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, latent_dim))

    def forward(self, particles):
        mask = torch.any(particles != 0, dim=-1, keepdim=True)
        return self.rho((self.phi(particles) * mask).sum(dim=1))


class DeepSets(nn.Module):
    def __init__(self, particle_features):
        super().__init__()
        self.encoder = DeepSetsEncoder(particle_features)
        self.classifier = nn.Sequential(nn.ReLU(), nn.Linear(24, 1))

    def forward(self, particles, event_features):
        return self.classifier(self.encoder(particles))


class HybridDeepSets(nn.Module):
    def __init__(self, particle_features, event_features):
        super().__init__()
        self.encoder = DeepSetsEncoder(particle_features)
        self.classifier = DNN(event_features + 24).network

    def forward(self, particles, event_features):
        return self.classifier(torch.cat((self.encoder(particles), event_features), dim=1))

## Select and inspect a model

Modify the constructors or add a new registry entry for architecture studies. The following cell creates a fresh model every time it is run.

In [ ]:
MODEL_BUILDERS = {
    "dnn": lambda: DNN(X_DNN.shape[1]),
    "gnn": lambda: GNN(MAX_TRACKS, X_PARTICLES.shape[2]),
    "gdnn": lambda: GDNN(MAX_TRACKS, X_PARTICLES.shape[2], X_DNN.shape[1]),
    "transformer": lambda: Transformer(X_PARTICLES.shape[2]),
    "tdnn": lambda: TDNN(X_PARTICLES.shape[2], X_DNN.shape[1]),
    "deepsets": lambda: DeepSets(X_PARTICLES.shape[2]),
    "hybrid_deepsets": lambda: HybridDeepSets(X_PARTICLES.shape[2], X_DNN.shape[1]),
}

if MODEL_NAME not in MODEL_BUILDERS:
    raise ValueError(f"Unknown model {MODEL_NAME!r}. Choose from {list(MODEL_BUILDERS)}")

model = MODEL_BUILDERS[MODEL_NAME]().to(DEVICE)
print(model)
print("Trainable parameters:", sum(parameter.numel() for parameter in model.parameters()))

## Training

The loop uses weighted binary cross-entropy, validation-loss scheduling, and early stopping. Hyperparameters remain visible in `configs/pipeline.toml`, while experimental changes can be made directly here.

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=float(TRAINING["learning_rate"]),
    weight_decay=float(TRAINING["weight_decay"]),
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.2, patience=2
)
loss_function = nn.BCEWithLogitsLoss(reduction="none")
grad_scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")

best_validation_loss = float("inf")
best_state = None
epochs_without_improvement = 0
history = []

for epoch in range(1, int(TRAINING["epochs"]) + 1):
    model.train()
    running_train_loss = 0
    train_count = 0

    for particles, event_features, labels, weights, _ in train_loader:
        particles = particles.to(DEVICE, non_blocking=True)
        event_features = event_features.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        weights = weights.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        autocast = torch.autocast("cuda") if DEVICE.type == "cuda" else nullcontext()
        with autocast:
            logits = model(particles, event_features)
            loss = (loss_function(logits, labels) * weights).mean()

        grad_scaler.scale(loss).backward()
        grad_scaler.step(optimizer)
        grad_scaler.update()
        running_train_loss += loss.item() * len(labels)
        train_count += len(labels)

    model.eval()
    running_validation_loss = 0
    validation_count = 0
    with torch.no_grad():
        for particles, event_features, labels, weights, _ in validation_loader:
            particles = particles.to(DEVICE)
            event_features = event_features.to(DEVICE)
            labels = labels.to(DEVICE)
            weights = weights.to(DEVICE)
            loss = (loss_function(model(particles, event_features), labels) * weights).mean()
            running_validation_loss += loss.item() * len(labels)
            validation_count += len(labels)

    train_loss = running_train_loss / train_count
    validation_loss = running_validation_loss / validation_count
    scheduler.step(validation_loss)
    history.append((epoch, train_loss, validation_loss))
    print(f"{epoch:03d} train={train_loss:.6f} validation={validation_loss:.6f}")

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_state = {name: value.detach().cpu() for name, value in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= int(TRAINING["patience"]):
            print("Early stopping")
            break

model.load_state_dict(best_state)
history = pd.DataFrame(history, columns=["epoch", "train_loss", "validation_loss"])
history.plot(x="epoch", y=["train_loss", "validation_loss"], figsize=(8, 5));

## Evaluation

Only the held-out test split is used for the reported classification metrics and ROC curves. The full selected sample is scored later for kinematic and binned studies.

In [ ]:
def predict(model, loader):
    model.eval()
    scores, labels, weights, indices = [], [], [], []
    with torch.no_grad():
        for particles, event_features, target, event_weight, source_index in loader:
            particles = particles.to(DEVICE)
            event_features = event_features.to(DEVICE)
            autocast = torch.autocast("cuda") if DEVICE.type == "cuda" else nullcontext()
            with autocast:
                probability = torch.sigmoid(model(particles, event_features))
            scores.append(probability.cpu().numpy().ravel())
            labels.append(target.numpy().ravel())
            weights.append(event_weight.numpy().ravel())
            indices.append(source_index.numpy().ravel())
    return tuple(np.concatenate(items) for items in (scores, labels, weights, indices))


test_scores, test_labels, test_weights, test_source_indices = predict(model, test_loader)
fpr, tpr, thresholds = roc_curve(
    test_labels, test_scores, sample_weight=test_weights
)
weighted_auc = auc(fpr, tpr)

print(classification_report(
    test_labels,
    test_scores >= 0.5,
    sample_weight=test_weights,
    target_names=["Background", "Signal"],
))
print("Weighted test AUC:", weighted_auc)

figure, axes = plt.subplots(1, 3, figsize=(17, 5))
axes[0].plot(fpr, tpr, label=f"AUC = {weighted_auc:.4f}")
axes[0].plot([0, 1], [0, 1], "--", color="0.6")
axes[0].set(xlabel="Background efficiency", ylabel="Signal efficiency", title="Weighted ROC")
axes[0].legend()

rejection = np.divide(1, fpr, out=np.full_like(fpr, np.inf), where=fpr > 0)
axes[1].plot(tpr, rejection)
axes[1].set(
    xlabel="Signal efficiency",
    ylabel="Background rejection",
    title="Efficiency versus rejection",
    yscale="log",
)

matrix = confusion_matrix(
    test_labels, test_scores >= 0.5, sample_weight=test_weights
)
sns.heatmap(matrix, annot=True, fmt=".1f", cmap="Blues", ax=axes[2])
axes[2].set(xlabel="Predicted", ylabel="True", title="Weighted confusion matrix")
figure.tight_layout()

## Score the complete selected dataset

The saved table includes a split label so downstream studies can distinguish unbiased test predictions from training and validation events.

In [ ]:
full_scores, _, full_weights, full_source_indices = predict(model, full_loader)
scored_events = combined.iloc[full_source_indices].copy().reset_index(drop=True)
scored_events["event_weight"] = full_weights
scored_events["score"] = full_scores

split_names = np.full(len(combined), "train", dtype=object)
split_names[validation_indices] = "validation"
split_names[test_indices] = "test"
scored_events["split"] = split_names[full_source_indices]

scored_events[["sample", "label", "score", "split"]].head()

## Extended plotting toolbox

The archived analysis used several specialized diagnostic plots in addition to the core ROC and confusion-matrix panel. The helpers below restore those plot families without producing every variable combination automatically. For reported model performance, pass only the held-out test rows.

In [ ]:
def weighted_quantile(values, quantile, weights):
    order = np.argsort(values)
    values = np.asarray(values)[order]
    weights = np.asarray(weights)[order]
    cumulative = np.cumsum(weights) / weights.sum()
    return np.interp(quantile, cumulative, values)


def weighted_efficiency_and_error(passed, weights):
    weights = np.asarray(weights, dtype=float)
    if len(weights) == 0 or weights.sum() == 0:
        return np.nan, np.nan
    efficiency = weights[np.asarray(passed)].sum() / weights.sum()
    effective_count = weights.sum() ** 2 / np.square(weights).sum()
    error = np.sqrt(efficiency * (1 - efficiency) / effective_count)
    return efficiency, error


def plot_score_distribution(frame, score="score", bins=60, density=True):
    figure, axis = plt.subplots(figsize=(8, 5))
    for label, name, color in ((0, "Background", "tab:blue"), (1, "Signal", "tab:orange")):
        subset = frame[frame["label"] == label]
        axis.hist(
            subset[score], bins=bins, weights=subset["event_weight"],
            density=density, histtype="step", linewidth=2, label=name, color=color,
        )
    axis.set(xlabel=score, ylabel="Density" if density else "Weighted events", title="Classifier score")
    axis.legend()
    return figure


test_events = scored_events[scored_events["split"] == "test"].copy()
plot_score_distribution(test_events);

In [ ]:
def plot_signal_efficiency_vs_feature(
    frame, feature, background_acceptances=(0.001, 0.005), bins=15, score="score"
):
    signal = frame[frame["label"] == 1]
    background = frame[frame["label"] == 0]
    edges = np.linspace(frame[feature].quantile(0.01), frame[feature].quantile(0.99), bins + 1)
    centers = (edges[:-1] + edges[1:]) / 2
    figure, axis = plt.subplots(figsize=(8, 5))

    for acceptance in background_acceptances:
        threshold = weighted_quantile(
            background[score], 1 - acceptance, background["event_weight"]
        )
        efficiencies, errors = [], []
        for low, high in zip(edges[:-1], edges[1:]):
            subset = signal[(signal[feature] >= low) & (signal[feature] < high)]
            efficiency, error = weighted_efficiency_and_error(
                subset[score] >= threshold, subset["event_weight"]
            )
            efficiencies.append(efficiency)
            errors.append(error)
        axis.errorbar(
            centers, efficiencies, yerr=errors, marker="o", capsize=3,
            label=f"{100 * acceptance:.2g}% background acceptance",
        )

    axis.set(xlabel=feature, ylabel="Signal efficiency", title=f"Signal efficiency vs {feature}")
    axis.legend()
    return figure


def plot_background_rejection_vs_feature(
    frame, feature, signal_efficiencies=(0.2, 0.4, 0.6), bins=15, score="score"
):
    signal = frame[frame["label"] == 1]
    background = frame[frame["label"] == 0]
    edges = np.linspace(background[feature].quantile(0.01), background[feature].quantile(0.99), bins + 1)
    centers = (edges[:-1] + edges[1:]) / 2
    figure, axis = plt.subplots(figsize=(8, 5))

    for target_efficiency in signal_efficiencies:
        threshold = weighted_quantile(
            signal[score], 1 - target_efficiency, signal["event_weight"]
        )
        rejection, rejection_error = [], []
        for low, high in zip(edges[:-1], edges[1:]):
            subset = background[(background[feature] >= low) & (background[feature] < high)]
            acceptance, error = weighted_efficiency_and_error(
                subset[score] >= threshold, subset["event_weight"]
            )
            if acceptance and np.isfinite(acceptance):
                rejection.append(1 / acceptance)
                rejection_error.append(error / acceptance ** 2)
            else:
                rejection.append(np.nan)
                rejection_error.append(np.nan)
        axis.errorbar(
            centers, rejection, yerr=rejection_error, marker="o", capsize=3,
            label=f"{100 * target_efficiency:.0f}% signal efficiency",
        )

    axis.set(
        xlabel=feature, ylabel="Background rejection",
        title=f"Background rejection vs {feature}", yscale="log",
    )
    axis.legend()
    return figure


# Examples for the system-pT or Higgs-pT columns used in the archived studies:
# plot_signal_efficiency_vs_feature(test_events, "system_pt")
# plot_background_rejection_vs_feature(test_events, "system_pt")

In [ ]:
def plot_feature_comparison(
    first, second, feature, labels=("First", "Second"), bins=50, log=False
):
    values = pd.concat([first[feature], second[feature]]).replace([np.inf, -np.inf], np.nan).dropna()
    edges = np.linspace(values.quantile(0.01), values.quantile(0.99), bins + 1)
    first_counts, _ = np.histogram(first[feature], bins=edges, density=True)
    second_counts, _ = np.histogram(second[feature], bins=edges, density=True)
    centers = (edges[:-1] + edges[1:]) / 2

    figure, (axis, ratio_axis) = plt.subplots(
        2, 1, figsize=(8, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]}
    )
    axis.step(centers, first_counts, where="mid", label=labels[0])
    axis.step(centers, second_counts, where="mid", label=labels[1])
    ratio = np.divide(first_counts, second_counts, out=np.full_like(first_counts, np.nan), where=second_counts > 0)
    ratio_axis.step(centers, ratio, where="mid")
    ratio_axis.axhline(1, color="0.5", linestyle="--")
    axis.set(ylabel="Density", title=f"Distribution comparison: {feature}")
    ratio_axis.set(xlabel=feature, ylabel=f"{labels[0]} / {labels[1]}")
    if log:
        axis.set_yscale("log")
    axis.legend()
    figure.tight_layout()
    return figure


def plot_2d_distribution(frame, x, y, bins=60, log_counts=True):
    from matplotlib.colors import LogNorm
    clean = frame[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    figure, axis = plt.subplots(figsize=(7, 6))
    norm = LogNorm() if log_counts else None
    histogram = axis.hist2d(clean[x], clean[y], bins=bins, norm=norm)
    figure.colorbar(histogram[3], ax=axis, label="Events")
    axis.set(xlabel=x, ylabel=y, title=f"{y} vs {x}")
    return figure


def plot_learning_curve(results, size="training_events", metric="test_auc"):
    ordered = results.sort_values(size)
    figure, axis = plt.subplots(figsize=(8, 5))
    axis.plot(ordered[size], ordered[metric], marker="o")
    axis.set(xlabel="Number of training events", ylabel=metric, title="Learning curve")
    return figure


def plot_cutflow(cut_counts, title="Event selection cut flow"):
    names = list(cut_counts)
    counts = np.asarray(list(cut_counts.values()))
    figure, axis = plt.subplots(figsize=(9, 5))
    axis.bar(names, counts)
    axis.set(ylabel="Events", title=title, yscale="log")
    axis.tick_params(axis="x", rotation=30)
    figure.tight_layout()
    return figure


# Truth/reconstruction constituent comparison example:
# plot_feature_comparison(reco_test, truth_test, "fj_track1dPhi", labels=("Reco", "Truth"))

## Save the experiment

Run names are isolated by domain and model. Existing runs are not replaced automatically. Set `OVERWRITE_RUN = True` only when replacement is intentional.

In [ ]:
OVERWRITE_RUN = False

runs_root = resolve_project_path(CONFIG["paths"]["runs"])
run_directory = runs_root / DOMAIN / MODEL_NAME / RUN_NAME
if run_directory.exists():
    if not OVERWRITE_RUN:
        raise FileExistsError(f"Run already exists: {run_directory}")
    shutil.rmtree(run_directory)
run_directory.mkdir(parents=True)

torch.save(
    {
        "model_name": MODEL_NAME,
        "state_dict": best_state,
        "max_tracks": MAX_TRACKS,
        "dnn_features": DNN_FEATURES,
        "particle_features": PARTICLE_FEATURES,
    },
    run_directory / "best_model.pt",
)
np.savez(
    run_directory / "preprocessing.npz",
    dnn_mean=dnn_scaler.mean_,
    dnn_scale=dnn_scaler.scale_,
    particle_mean=particle_scaler.mean_,
    particle_scale=particle_scaler.scale_,
    train_indices=train_indices,
    validation_indices=validation_indices,
    test_indices=test_indices,
)
history.to_csv(run_directory / "training_history.csv", index=False)

test_predictions = scored_events[scored_events["split"] == "test"].copy()
test_predictions.to_parquet(run_directory / "test_predictions.parquet", index=False)
scored_events.to_parquet(run_directory / "full_predictions.parquet", index=False)

metrics = {
    "domain": DOMAIN,
    "model": MODEL_NAME,
    "run_name": RUN_NAME,
    "weighted_test_auc": float(weighted_auc),
    "best_validation_loss": float(best_validation_loss),
    "selected_events": len(combined),
    "test_events": len(test_predictions),
}
(run_directory / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
print("Saved:", run_directory)

## Comparing saved runs

This final helper is useful for truth/reconstruction or architecture comparisons. It only reads the held-out test predictions from each run.

In [ ]:
def compare_saved_runs(run_directories):
    figure, (roc_axis, rejection_axis) = plt.subplots(1, 2, figsize=(13, 5))
    for directory in map(Path, run_directories):
        predictions = pd.read_parquet(directory / "test_predictions.parquet")
        fpr, tpr, _ = roc_curve(
            predictions["label"],
            predictions["score"],
            sample_weight=predictions["event_weight"],
        )
        run_auc = auc(fpr, tpr)
        label = f"{directory.parent.name}/{directory.name} ({run_auc:.4f})"
        roc_axis.plot(fpr, tpr, label=label)
        rejection_axis.plot(
            tpr,
            np.divide(1, fpr, out=np.full_like(fpr, np.inf), where=fpr > 0),
            label=label,
        )

    roc_axis.plot([0, 1], [0, 1], "--", color="0.6")
    roc_axis.set(xlabel="Background efficiency", ylabel="Signal efficiency", title="Weighted ROC")
    rejection_axis.set(
        xlabel="Signal efficiency",
        ylabel="Background rejection",
        title="Efficiency versus rejection",
        yscale="log",
    )
    for axis in (roc_axis, rejection_axis):
        axis.legend()
        axis.grid(True, which="both", alpha=0.3)
    figure.tight_layout()
    return figure


# Example:
# compare_saved_runs([
#     PROJECT_ROOT / "outputs/runs/truth/gdnn/baseline_v1",
#     PROJECT_ROOT / "outputs/runs/reco/gdnn/baseline_v1",
# ])